# C9-dimensionality-reduction — Practice p13 — Solution

In [1]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
centers = rng.normal(0.0, 2.5, (3, 10))
labels = np.repeat(np.arange(3), 50)
X = centers[labels] + rng.normal(0.0, 0.9, (150, 10))

mu = X.mean(axis=0)
Xc = X - mu
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
assert np.max(np.abs(Xc.mean(axis=0))) < 1e-10
total_var = float(X.var(axis=0, ddof=1).sum())
total_var_svd = float((s**2).sum() / (X.shape[0] - 1))
assert abs(total_var - total_var_svd) < 1e-9

evr = s**2 / (s**2).sum()
cum = np.cumsum(evr)
r90 = int(np.argmax(cum >= 0.90) + 1)
assert cum[r90 - 1] >= 0.90
assert r90 == 1 or cum[r90 - 2] < 0.90

Vfix = Vt * np.sign(Vt[np.arange(10), np.argmax(np.abs(Vt), axis=1)])[:, None]
T2 = Xc @ Vfix[:2].T
pairs = ((0, 1), (0, 2), (1, 2))
d_full = np.empty(3, dtype=np.float64)
d_proj = np.empty(3, dtype=np.float64)
for q, (a, b) in enumerate(pairs):
    delta_full = X[labels == a].mean(axis=0) - X[labels == b].mean(axis=0)
    delta_proj = T2[labels == a].mean(axis=0) - T2[labels == b].mean(axis=0)
    d_full[q] = np.sqrt((delta_full * delta_full).sum())
    d_proj[q] = np.sqrt((delta_proj * delta_proj).sum())
worst_shrink = float(np.max((d_full - d_proj) / d_full))

intervals = np.empty((3, 2), dtype=np.float64)
cluster_means = np.empty(3, dtype=np.float64)
for cluster in range(3):
    scores = T2[labels == cluster, 0]
    intervals[cluster] = (scores.min(), scores.max())
    cluster_means[cluster] = scores.mean()
order = np.argsort(cluster_means)
gaps = intervals[order[1:], 0] - intervals[order[:-1], 1]
gap_index = int(np.argmin(gaps))
min_gap = float(gaps[gap_index])
crowded_pair = tuple(sorted((int(order[gap_index]), int(order[gap_index + 1]))))

print("EVR / cumulative:", evr, cum)
print("r90:", r90)
print("full / projected centroid distances:", d_full, d_proj)
print("worst shrink:", worst_shrink)
print("PC1 intervals / closest pair:", intervals, crowded_pair, min_gap)

EVR / cumulative: [0.60273738 0.28657469 0.01876251 0.01786202 0.01495694 0.01458541
 0.01347792 0.01250499 0.00945559 0.00908256] [0.60273738 0.88931207 0.90807458 0.9259366  0.94089355 0.95547895
 0.96895687 0.98146186 0.99091744 1.        ]
r90: 3
full / projected centroid distances: [10.18422535 14.02877572 11.88233125] [10.18340661 14.02852392 11.88199254]
worst shrink: 8.039375517466078e-05
PC1 intervals / closest pair: [[ 4.16396973  8.15088453]
 [-0.53646105  3.75129788]
 [-9.53721385 -4.89835876]] (0, 1) 0.41267184184426675


Clusters $0$ and $1$ nearly merge on PC1: their intervals have only a $0.4126718$ gap.  PC2 raises cumulative explained variance from $0.6027374$ to $0.8893121$ and makes every centroid-pair shrink smaller than $8.04\times10^{-5}$, although a third component is narrowly required to reach $90\%$.

### Answer check

In [2]:
assert mu.shape == (10,) and Xc.shape == (150, 10)
assert np.max(np.abs(Xc.mean(axis=0))) < 1e-10
assert abs(total_var - total_var_svd) < 1e-9
assert r90 == 3
assert np.isclose(cum[1], 0.88931207, atol=1e-8, rtol=0)
assert np.isclose(cum[2], 0.90807458, atol=1e-8, rtol=0)
assert worst_shrink < 1e-4
assert np.isclose(min_gap, 0.4126718418442654, atol=1e-12, rtol=0)
assert crowded_pair == (0, 1)